# 01 — Validate exports

Validate source shapes, label completeness, and the paired missingness behavior of `R_x` / `N_x`.

In [1]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXPORT_DIR = Path("../exports")
assert EXPORT_DIR.exists(), f"Expected exports directory at {EXPORT_DIR.resolve()}"

def load_csv(name):
    path = EXPORT_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing required export: {path}")
    return pd.read_csv(path)

def drop_export_index(df):
    return df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")], errors="ignore")

def attr_id(variable):
    return int(str(variable).split("_")[-1])

def canonical_pattern(serialized):
    return tuple(sorted(json.loads(serialized)))

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)


In [2]:
files = sorted(p.name for p in EXPORT_DIR.glob("*.csv"))
print(f"Found {len(files)} CSV exports")
for f in files:
    print(" -", f)

Found 19 CSV exports
 - null_column_summary.csv
 - null_pattern_details.csv
 - smart_2018_failure_coverage.csv
 - smart_2018_mc1_feature_availability.csv
 - smart_2018_mc1_feature_missingness_risk.csv
 - smart_2018_mc1_null_pattern_failure_risk.csv
 - smart_2018_mc1_null_patterns.csv
 - smart_2018_mc1_year_comparison.csv
 - smart_2018_model_null_patterns.csv
 - smart_2019_mc1_feature_availability.csv
 - smart_2019_mc1_null_pattern_failure_risk.csv
 - smart_2019_mc1_null_patterns.csv
 - smart_2019_mc1_structurally_unavailable.csv
 - smart_2019_mc1_year_comparison.csv
 - smart_2019_model_null_patterns.csv
 - smart_2019_null_combinations.csv
 - smart_2019_null_counts_by_column.csv
 - ssd_failure_labels_null_combinations.csv
 - ssd_failure_labels_null_counts_by_column.csv


## Failure-label null audit

In [3]:
fail_comb = load_csv("ssd_failure_labels_null_combinations.csv")
fail_cols = load_csv("ssd_failure_labels_null_counts_by_column.csv")
display(fail_comb)
print("Columns with missing failure-label values:", len(fail_cols))
assert len(fail_comb) == 1
assert int(fail_comb.loc[0, "NULL_COLUMN_COUNT"]) == 0
assert int(fail_comb.loc[0, "RECORD_COUNT"]) == 16305
assert fail_cols.empty

,NULL_COLUMN_COUNT,NULL_COLUMNS,RECORD_COUNT
0,0,[],16305


Columns with missing failure-label values: 0


## Validate R/N paired missingness from exact null patterns

In [4]:
p18 = drop_export_index(load_csv("null_pattern_details.csv"))
p19 = load_csv("smart_2019_null_combinations.csv")

def pair_mismatch_summary(patterns):
    bad_patterns = 0
    bad_records = 0
    bad_attrs = set()
    for row in patterns.itertuples(index=False):
        cols = set(json.loads(row.NULL_COLUMNS))
        ids = {attr_id(c) for c in cols if re.match(r"^[RN]_\d+$", c)}
        bad = [a for a in ids if ((f"R_{a}" in cols) != (f"N_{a}" in cols))]
        if bad:
            bad_patterns += 1
            bad_records += int(row.RECORD_COUNT)
            bad_attrs.update(bad)
    return bad_patterns, bad_records, sorted(bad_attrs)

for year, df in [(2018,p18),(2019,p19)]:
    result = pair_mismatch_summary(df)
    print(year, {"mismatched_patterns":result[0], "affected_records":result[1], "attributes":result[2]})
    assert result[0] == 0

2018 {'mismatched_patterns': 0, 'affected_records': 0, 'attributes': []}


2019 {'mismatched_patterns': 0, 'affected_records': 0, 'attributes': []}


**Interpretation.** In the exported pattern universe, `R_x` and `N_x` never disappear independently. This supports treating missingness as paired at the SMART-attribute level. It does not establish the numerical mapping between raw and normalized values.